# Microsoft Foundry · Agente com automação de navegador (Browser Automation)

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 19): um agente que **controla um navegador real** (abre sites, digita, clica e lê a tela) usando a ferramenta **Browser Automation**, com os navegadores fornecidos pelo **Azure Playwright Workspace**.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Pré-requisitos: Playwright Workspace e conexão no Foundry](#1.-Pré-requisitos:-Playwright-Workspace-e-conexão-no-Foundry)
2. [Instalação das bibliotecas](#2.-Instalação-das-bibliotecas)
3. [Variáveis de ambiente](#3.-Variáveis-de-ambiente)
4. [Conectando ao projeto do Foundry](#4.-Conectando-ao-projeto-do-Foundry)
5. [Cliente OpenAI do projeto](#5.-Cliente-OpenAI-do-projeto)
6. [Obtendo o ID da conexão Browser Automation](#6.-Obtendo-o-ID-da-conexão-Browser-Automation)
7. [Montando a ferramenta de automação](#7.-Montando-a-ferramenta-de-automação)
8. [Criando o agente](#8.-Criando-o-agente)
9. [Criando a conversa](#9.-Criando-a-conversa)
10. [Enviando a tarefa ao agente](#10.-Enviando-a-tarefa-ao-agente)
11. [Verificando no portal do Foundry](#11.-Verificando-no-portal-do-Foundry)
12. [Resumo dos aprendizados](#12.-Resumo-dos-aprendizados)
13. [Limpeza e custos](#13.-Limpeza-e-custos)
14. [Interface com Streamlit](#14.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Muitas informações só estão disponíveis **na tela de um site**, sem API: é preciso abrir a página, pesquisar, clicar em botões e ler o resultado. Fazer isso manualmente e de forma repetitiva é lento; programar um robô (*scraper*) para cada site é frágil.

### 0.2 Objetivo

- Criar um **Azure Playwright Workspace**, que fornece navegadores remotos em sandbox.
- Conectar esse navegador ao **Microsoft Foundry** como a ferramenta **Browser Automation**.
- Criar um agente que recebe uma tarefa em linguagem natural e **executa os cliques** sozinho: descobrir a variação **YTD** (no ano) das ações da Microsoft no Yahoo Finance.
- Entender as **limitações** (latência, CAPTCHA, necessidade de instruções precisas).

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Azure Playwright Workspace** | Fornece navegadores reais em sandbox (também usado em testes automatizados de interface). |
| **Microsoft Foundry** (projeto) | Guarda a **conexão** Browser Automation (endpoint + token), o agente e os traces. |
| **Foundry Agent Service** | Hospeda o `browser-automation-agent` e aciona a ferramenta. |
| **Browser Automation tool** (`BrowserAutomationAgentTool`) | Ferramenta (em *preview*) que transforma as decisões do agente em ações no navegador. |
| **Azure OpenAI** (deployment GPT-4) | Interpreta a tarefa, decide as ações e resume o resultado. |
| **Microsoft Entra ID** | Autenticação sem chave (`DefaultAzureCredential`). |

### 0.4 Como funciona (visão simples)

![Fluxo do agente com automação de navegador](fluxo_browser_automation.svg)

### 0.5 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

> 💡 **Para um leigo:** é como pedir a um estagiário: "abra o site X, digite Y na busca, clique em Z e me diga o número que aparecer". O agente faz exatamente isso, mas em um computador na nuvem.

---
## 1. Pré-requisitos: Playwright Workspace e conexão no Foundry

### 1.1 Criar o Azure Playwright Workspace

1. `portal.azure.com` → pesquisar **Playwright Workspaces** → **Create**.
2. Informar o **grupo de recursos** e o **nome do workspace**.
3. O serviço começa com um **período de avaliação gratuito**; depois passa a cobrar **por uso** (só enquanto o navegador é usado). Os valores podem ser simulados na **Calculadora de Preços do Azure**.
4. Na página **Overview** do workspace há dois valores: o *API base endpoint* e o **Browser endpoint**. É o **Browser endpoint** que vamos usar.

### 1.2 Gerar o token de acesso

1. No workspace → **Settings → Access Management**.
2. **Generate token** (Playwright Service Access Token), com nome e validade (ex.: 7 dias).

> ⚠️ O token **só aparece uma vez**: copie e guarde em local seguro. Ele dá acesso aos navegadores do workspace e **não deve ir para o código nem para o GitHub**.

### 1.3 Criar a conexão no Foundry

1. Portal do Foundry → **Build → Tools → Connect tool**.
2. Aba **Configured** → **Browser Automation** → **Add tool** → **nova conexão**.
3. Preencher o **nome da conexão** (vai no `.env` como `BROWSER_AUTOMATION_CONNECTION_NAME`), o **Browser endpoint** do Playwright Workspace e o **token de acesso**.
4. **Connect**.

O token fica guardado **na conexão do Foundry**; o código só precisa do **nome** da conexão.

---
## 2. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |


> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 3. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |
| `BROWSER_AUTOMATION_CONNECTION_NAME` | Nome da conexão Browser Automation. | Portal do Foundry → **Tools** (criada na seção 1.3). |

```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>
BROWSER_AUTOMATION_CONNECTION_NAME=<browser-automation-connection-name>
```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

Classes importadas para este lab:

| Classe | Para que serve |
| --- | --- |
| `BrowserAutomationAgentTool` | A ferramenta de automação de navegador do agente. |
| `BrowserAutomationToolParameters` | Parâmetros da ferramenta (versão *preview*). |
| `BrowserAutomationToolConnectionParameters` | Qual conexão do projeto usar (`project_connection_id`). |
| `WebSearchPreviewTool`, `ApproximateLocation` | Importadas, mas não usadas neste lab. |

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# BROWSER_AUTOMATION_CONNECTION_NAME = nome da conexão criada no portal
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WebSearchPreviewTool, ApproximateLocation
from azure.ai.projects.models import BrowserAutomationToolConnectionParameters, BrowserAutomationToolParameters, BrowserAutomationAgentTool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
browser_automation_connection_name = os.getenv("BROWSER_AUTOMATION_CONNECTION_NAME")

---
## 4. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 5. Cliente OpenAI do projeto

O agente roda sobre um modelo **GPT** (Azure OpenAI). Para conversar com ele usamos a **Responses API** do SDK `openai`, e o próprio projeto entrega esse cliente já configurado com o endpoint e a autenticação do Foundry: `project_client.get_openai_client()`.

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# É ele que cria conversas e envia mensagens ao agente (Responses API)
openai_client = project_client.get_openai_client()

---
## 6. Obtendo o ID da conexão Browser Automation

O mesmo padrão do Lab 17 (MCP): percorremos `project_client.connections.list()` e guardamos o `id` da conexão cujo nome é igual a `BROWSER_AUTOMATION_CONNECTION_NAME`. É esse ID que autoriza o agente a usar o navegador.

In [ ]:
# Procurando a conexão Browser Automation pelo nome e guardando o seu ID
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == browser_automation_connection_name:
        connection_id = connection.id
        break

print(f"The Browser Automation Tool Connection ID is: {connection_id}")

---
## 7. Montando a ferramenta de automação

A ferramenta é montada em camadas:

```
BrowserAutomationAgentTool
└── browser_automation_preview = BrowserAutomationToolParameters
    └── connection = BrowserAutomationToolConnectionParameters
        └── project_connection_id = connection_id
```

Ou seja: "use a automação de navegador (*preview*) pela conexão X do projeto".

In [ ]:
# Montando a ferramenta de automação de navegador apontando para a conexão do projeto
tool = BrowserAutomationAgentTool(
    browser_automation_preview=BrowserAutomationToolParameters(
        connection = BrowserAutomationToolConnectionParameters(
            project_connection_id=connection_id
        )
    )
)

---
## 8. Criando o agente

| Campo | Valor |
| --- | --- |
| `agent_name` | `browser-automation-agent` |
| `instructions` | *"Você é um assistente útil que pode automatizar tarefas no navegador web usando a ferramenta Browser Automation."* |
| `tools` | `[tool]` |

In [ ]:
# Criando o agente com a ferramenta de automação de navegador
agent = project_client.agents.create_version(
    agent_name = "browser-automation-agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        tools=[tool],
        instructions="You are a helpful assistant that can automate web browser tasks using the Browser Automation Tool.",
    )
)

---
## 9. Criando a conversa

A **conversation** é um objeto guardado na Azure que **armazena o histórico** entre o usuário e o agente, identificado pelo `conversation.id`. Todas as mensagens enviadas com o mesmo id fazem parte do mesmo diálogo, então o agente "lembra" o que já foi dito.

> 💡 No portal, o mesmo `conversation.id` aparece em **Traces**, com cada chamada de ferramenta e o consumo de tokens.

In [ ]:
# Criando a conversa que guarda o histórico (memória) entre usuário e agente
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 10. Enviando a tarefa ao agente

Com automação de navegador, **o pedido precisa ser muito específico**: em vez de "qual a variação das ações da Microsoft?", descrevemos **cada passo da interface**. Em português:

> *Seu objetivo é informar a variação percentual do preço das ações da Microsoft no ano (YTD). Para isso, acesse **finance.yahoo.com**. No topo da página há uma **barra de pesquisa**: digite **'MSFT'**. No topo da página de resultado aparece um gráfico do preço da ação. Clique em **'YTD'** acima do gráfico e informe o **percentual** que aparece logo abaixo.*

> 💡 Instruções vagas deixam o agente "solto" demais: ele pode clicar no lugar errado, demorar mais e piorar a experiência. Controlar o comportamento do agente é o principal desafio dessa ferramenta.

In [ ]:
# Tarefa passo a passo: site, onde digitar, onde clicar e o que informar
user_query = """Your goal is to report the percent of Microsoft year-to-date stock price change.
            To do that, go to the website finance.yahoo.com.
            At the top of the page, you will find a search bar.
            Enter the value 'MSFT', to get information about the Microsoft stock price.
            At the top of the resulting page you will see a default chart of Microsoft stock price.
            Click on 'YTD' at the top of that chart, and report the percent value that shows up just below it."""

| Parâmetro | Valor | Para que serve |
| --- | --- | --- |
| `conversation` | `conversation.id` | Grava a mensagem na conversa e envia o histórico ao modelo. |
| `extra_body` → `agent` | `{"name": "browser-automation-agent", "type": "agent_reference"}` | Diz **qual agente** responde. Sem versão informada, usa **a mais recente**. |
| `input` | pergunta do usuário | O texto enviado ao agente. |

A resposta final em texto fica em **`response.output_text`**. Repare que a pergunta **não manda** o agente usar a ferramenta: é o próprio agente que analisa a intenção e decide.

> ⏱️ Esta célula pode levar **2 a 3 minutos**: o agente está operando um navegador de verdade.

In [ ]:
# Enviando a tarefa: o agente controla o navegador remoto e devolve o resultado
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body = {
        "agent": {
            "name": "browser-automation-agent",
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Agent Response: {response.output_text}")

### 10.1 O que aconteceu na demonstração da aula

- Na **primeira** execução o agente não encontrou o valor; na **segunda**, encontrou.
- Resposta: *"A variação percentual no ano (YTD) das ações da Microsoft, conforme exibido no Yahoo Finance, é de **14,3%**."*
- Cada execução levou **2 a 3 minutos**.
- Um teste extra ("pesquise cursos de Microsoft Foundry na Udemy e liste os 5 primeiros") **falhou**: o site bloqueou o acesso com **Cloudflare e CAPTCHA**. Nem todo site permite navegação automatizada.

---
## 11. Verificando no portal do Foundry

Em **Agentes → `browser-automation-agent` → Traces**, abrindo o `conversation.id`:

| Etapa | O que aconteceu |
| --- | --- |
| 1 | O agente analisou a tarefa e **chamou a função de automação do navegador**. |
| 2 | **Sintetizou** o que a ferramenta devolveu (o conteúdo da página). |
| 3 | Entregou a **resposta final** ao usuário. |

---
## 12. Resumo dos aprendizados

- A **Browser Automation** deixa o agente operar um navegador real: abrir sites, digitar, clicar e ler a tela.
- Os navegadores vêm do **Azure Playwright Workspace**, conectado ao Foundry por **endpoint + token** (guardados na conexão, fora do código).
- No código, o padrão é o mesmo de outras ferramentas conectadas: `connections.list()` → `connection.id` → ferramenta → agente.
- Funciona melhor com **instruções passo a passo**; é **lento** (minutos) e pode ser bloqueado por **CAPTCHA/Cloudflare**.
- Bom para **tarefas longas em segundo plano**, sem intervenção humana, e não para chat em tempo real.

---
## 13. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| Azure Playwright Workspace | Avaliação gratuita no início; depois, cobrança **por uso** dos navegadores. Apague o workspace no `portal.azure.com` ao terminar. |
| Token de acesso do Playwright | Sem custo, mas é uma credencial: defina validade curta e revogue em *Access Management* quando não precisar mais. |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 14. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` (seção 4) e `get_openai_client()` (seção 5) |
| `nova_conversa()` | `conversations.create` (seção 9) |
| `executar_tarefa()` | `responses.create` com `agent_reference` (seção 10) |
| `TAREFA_EXEMPLO` | a mesma tarefa passo a passo do Yahoo Finance (seção 10) |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso o cliente fica em cache (`@st.cache_resource`) e o `conversation.id` e os resultados ficam no `st.session_state`.
- Como a tarefa é um **texto longo e detalhado**, a interface usa um **formulário** (`st.form` + `text_area`) em vez de um chat.
- A barra lateral avisa sobre a **latência** (minutos por tarefa) e os bloqueios por CAPTCHA.

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
BROWSER_AUTOMATION_CONNECTION_NAME = os.getenv("BROWSER_AUTOMATION_CONNECTION_NAME")
AGENT_NAME = "browser-automation-agent"  # criado na seção 8 do notebook

# Mesma tarefa do notebook: instruções passo a passo funcionam melhor
TAREFA_EXEMPLO = """Your goal is to report the percent of Microsoft year-to-date stock price change.
To do that, go to the website finance.yahoo.com.
At the top of the page, you will find a search bar.
Enter the value 'MSFT', to get information about the Microsoft stock price.
At the top of the resulting page you will see a default chart of Microsoft stock price.
Click on 'YTD' at the top of that chart, and report the percent value that shows up just below it."""

st.set_page_config(page_title="Browser Automation Agent · Microsoft Foundry", page_icon="🖱️", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
@st.cache_resource
def criar_clientes():
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return project_client.get_openai_client()


# ------------------------------------------------------------------
# 3. Conversa e tarefa
# ------------------------------------------------------------------
def nova_conversa():
    st.session_state.conversation_id = criar_clientes().conversations.create().id
    st.session_state.historico = []


def executar_tarefa(tarefa):
    # Mesmo responses.create do notebook; pode levar alguns minutos
    return criar_clientes().responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=tarefa,
    )


def etapas_do_agente(response):
    # Etapas que não são mensagem: chamadas da ferramenta de automação do navegador
    return [f"{item.type} {getattr(item, 'name', '') or ''}".strip() for item in response.output if item.type != "message"]


# ------------------------------------------------------------------
# 4. Interface
# ------------------------------------------------------------------
st.title("🖱️ Agente que navega na web")
st.caption("Foundry Agent Service + Browser Automation (Azure Playwright Workspace): o agente abre sites, digita e clica por você.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("🖱️ Automação")
    st.caption(f"Conexão: `{BROWSER_AUTOMATION_CONNECTION_NAME}`  \nAgente: `{AGENT_NAME}`  \nConversa: `{st.session_state.conversation_id}`")
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()
    st.divider()
    st.warning("⏱️ Cada tarefa pode levar **2 a 3 minutos**. Sites com Cloudflare/CAPTCHA podem bloquear o agente.")

# A tarefa é um texto longo (passo a passo), então usamos um formulário em vez do chat_input
with st.form("tarefa"):
    tarefa = st.text_area("Descreva a tarefa passo a passo (site, onde digitar, onde clicar, o que informar)",
                          value=TAREFA_EXEMPLO, height=180)
    enviar = st.form_submit_button("▶️ Executar no navegador")

if enviar and tarefa.strip():
    with st.spinner("O agente está navegando... (pode levar alguns minutos)"):
        try:
            response = executar_tarefa(tarefa)
        except ClientAuthenticationError:
            st.error("Falha no login da Azure. Execute `az login` no terminal.")
        except (HttpResponseError, APIError) as erro:
            st.error(f"Erro ao executar a tarefa: {erro}")
        else:
            st.session_state.historico.insert(0, {"tarefa": tarefa, "resultado": response.output_text,
                                                  "etapas": etapas_do_agente(response)})

# Resultados (o mais recente primeiro)
for i, item in enumerate(st.session_state.historico):
    with st.container(border=True):
        st.markdown(f"**Resultado {len(st.session_state.historico) - i}**")
        st.markdown(item["resultado"])
        if item["etapas"]:
            st.caption("🔧 " + " → ".join(item["etapas"]))
        with st.expander("Ver a tarefa enviada"):
            st.text(item["tarefa"])
